In [6]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

# Load and Sort Data

In [5]:
# Load CSV and parse event times
df = pd.read_csv(
    'first_25000_rows.csv',
    parse_dates=['ts_event']
)

# Ensure events are in chronological order for each symbol
df.sort_values(['symbol', 'ts_event'], inplace=True)
df


,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
0,2024-10-21T11:54:29.221230963Z,2024-10-21 11:54:29.221064336+00:00,10,2,38,C,B,1,233.62,2,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
1,2024-10-21T11:54:29.223936626Z,2024-10-21 11:54:29.223769812+00:00,10,2,38,A,B,0,233.67,2,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
2,2024-10-21T11:54:29.225196809Z,2024-10-21 11:54:29.225030400+00:00,10,2,38,A,B,0,233.67,3,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
3,2024-10-21T11:54:29.712600612Z,2024-10-21 11:54:29.712434212+00:00,10,2,38,A,B,2,233.52,200,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
4,2024-10-21T11:54:29.764839221Z,2024-10-21 11:54:29.764673165+00:00,10,2,38,C,B,2,233.52,200,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,2024-10-21T13:04:16.583694069Z,2024-10-21 13:04:16.583527688+00:00,10,2,38,A,B,2,233.46,200,...,105,1,2,233.25,234.50,55,63,2,4,AAPL
4996,2024-10-21T13:04:17.976627074Z,2024-10-21 13:04:17.976461017+00:00,10,2,38,A,A,1,233.69,200,...,105,1,2,233.25,234.50,55,63,2,4,AAPL
4997,2024-10-21T13:04:20.085804687Z,2024-10-21 13:04:20.085638629+00:00,10,2,38,C,B,2,233.46,200,...,105,2,2,233.24,234.50,1,63,1,4,AAPL
4998,2024-10-21T13:04:20.085817362Z,2024-10-21 13:04:20.085651109+00:00,10,2,38,A,B,3,233.44,200,...,105,1,2,233.25,234.50,55,63,2,4,AAPL


# Compute Per-Level Order-Flow Increments

In [8]:
def compute_order_flows(group, max_level=10):
    """
    For each level m, compute:
      OF_b_m: bid-side flow increment
      OF_a_m: ask-side flow increment
    according to price up/same/down rules.
    """
    for m in range(max_level):
        # current & previous price/size
        pb = group[f'bid_px_{m:02d}']; qb = group[f'bid_sz_{m:02d}']
        pa = group[f'ask_px_{m:02d}']; qa = group[f'ask_sz_{m:02d}']
        pb_prev = pb.shift(); qb_prev = qb.shift()
        pa_prev = pa.shift(); qa_prev = qa.shift()

        # bid-side: price↑→+size, same→Δsize,↓→−size
        ofb = np.where(
            pb > pb_prev, qb,
            np.where(pb == pb_prev, qb - qb_prev, -qb)
        )

        # ask-side: price↑→−size, same→Δsize,↓→+size
        ofa = np.where(
            pa > pa_prev, -qa,
            np.where(pa == pa_prev, qa - qa_prev, qa)
        )

        group[f'OF_b_{m}'] = ofb
        group[f'OF_a_{m}'] = ofa

    return group

# Apply per-symbol
df = df.groupby('symbol', group_keys=False).apply(compute_order_flows)
df


,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,OF_b_5,OF_a_5,OF_b_6,OF_a_6,OF_b_7,OF_a_7,OF_b_8,OF_a_8,OF_b_9,OF_a_9
0,2024-10-21T11:54:29.221230963Z,2024-10-21 11:54:29.221064336+00:00,10,2,38,C,B,1,233.62,2,...,-110.0,25.0,-10.0,200.0,-110.0,44.0,-100.0,155.0,-55.0,400.0
1,2024-10-21T11:54:29.223936626Z,2024-10-21 11:54:29.223769812+00:00,10,2,38,A,B,0,233.67,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2024-10-21T11:54:29.225196809Z,2024-10-21 11:54:29.225030400+00:00,10,2,38,A,B,0,233.67,3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024-10-21T11:54:29.712600612Z,2024-10-21 11:54:29.712434212+00:00,10,2,38,A,B,2,233.52,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2024-10-21T11:54:29.764839221Z,2024-10-21 11:54:29.764673165+00:00,10,2,38,C,B,2,233.52,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,2024-10-21T13:04:16.583694069Z,2024-10-21 13:04:16.583527688+00:00,10,2,38,A,B,2,233.46,200,...,10.0,0.0,45.0,0.0,110.0,0.0,100.0,0.0,55.0,0.0
4996,2024-10-21T13:04:17.976627074Z,2024-10-21 13:04:17.976461017+00:00,10,2,38,A,A,1,233.69,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4997,2024-10-21T13:04:20.085804687Z,2024-10-21 13:04:20.085638629+00:00,10,2,38,C,B,2,233.46,200,...,-45.0,0.0,-110.0,0.0,-100.0,0.0,-55.0,0.0,-1.0,0.0
4998,2024-10-21T13:04:20.085817362Z,2024-10-21 13:04:20.085651109+00:00,10,2,38,A,B,3,233.44,200,...,10.0,0.0,45.0,0.0,110.0,0.0,100.0,0.0,55.0,0.0


# Build a Regular Timestamp Grid
We pull out the full span of ts_event and build, e.g., 1-minute grid points.

In [10]:
def choose_timestamps(df, freq='1T'):
    """
    Return a DateTimeIndex from min→max ts_event at given frequency.
    """
    idx = df['ts_event'].dt.round(freq)
    return pd.date_range(idx.min(), idx.max(), freq=freq)

timestamps = choose_timestamps(df, freq='1T')
timestamps

DatetimeIndex(['2024-10-21 11:54:00+00:00', '2024-10-21 11:55:00+00:00',
               '2024-10-21 11:56:00+00:00', '2024-10-21 11:57:00+00:00',
               '2024-10-21 11:58:00+00:00', '2024-10-21 11:59:00+00:00',
               '2024-10-21 12:00:00+00:00', '2024-10-21 12:01:00+00:00',
               '2024-10-21 12:02:00+00:00', '2024-10-21 12:03:00+00:00',
               '2024-10-21 12:04:00+00:00', '2024-10-21 12:05:00+00:00',
               '2024-10-21 12:06:00+00:00', '2024-10-21 12:07:00+00:00',
               '2024-10-21 12:08:00+00:00', '2024-10-21 12:09:00+00:00',
               '2024-10-21 12:10:00+00:00', '2024-10-21 12:11:00+00:00',
               '2024-10-21 12:12:00+00:00', '2024-10-21 12:13:00+00:00',
               '2024-10-21 12:14:00+00:00', '2024-10-21 12:15:00+00:00',
               '2024-10-21 12:16:00+00:00', '2024-10-21 12:17:00+00:00',
               '2024-10-21 12:18:00+00:00', '2024-10-21 12:19:00+00:00',
               '2024-10-21 12:20:00+00:00', '2024-1

# Compute OFI Features per Symbol & Timestamp
Iterate over symbols and timestamps, and for each 1-minute window ending at t:

Best-Level OFI = sum(OF_b_0 − OF_a_0)

Multi-Level OFI = normalized vector of (OF_b_m − OF_a_m) / QM

Integrated OFI = first PCA component of the 10-D vector

In [11]:
results = []

for sym, grp in df.groupby('symbol'):
    grp = grp.set_index('ts_event')
    multi_matrix = []

    for t in timestamps:
        window = grp[(grp.index > t - pd.Timedelta('1min')) & (grp.index <= t)]
        if window.empty:
            continue

        # 5.1 Best-Level OFI
        best = (window['OF_b_0'] - window['OF_a_0']).sum()

        # 5.2 Multi-Level OFI (normalized by average depth QM)
        depth_cols = [f'bid_sz_{m:02d}' for m in range(10)] + [f'ask_sz_{m:02d}' for m in range(10)]
        QM = window[depth_cols].sum(axis=1).mean() / (2 * 10)

        levels = [
            (window[f'OF_b_{m}'] - window[f'OF_a_{m}']).sum() / QM
            if QM != 0 else np.nan
            for m in range(10)
        ]

        multi_matrix.append(levels)

        row = {'symbol': sym, 'timestamp': t, 'best_ofi': best}
        row.update({f'ofi_lvl_{m+1}': levels[m] for m in range(10)})
        results.append(row)

    # 5.3 Integrated OFI via PCA
    if multi_matrix:
        X = np.array(multi_matrix)
        pca = PCA(n_components=1).fit(X)
        scores = pca.transform(X).flatten()
        # assign back
        start = len(results) - len(scores)
        for i, score in enumerate(scores, start=start):
            results[i]['integrated_ofi'] = float(score)


In [12]:
res_df = pd.DataFrame(results)
pivot = res_df.pivot(index='timestamp', columns='symbol', values='best_ofi')

cross_vals = []
for _, row in res_df.iterrows():
    sym, t = row['symbol'], row['timestamp']
    others = pivot.columns.difference([sym])
    cross_vals.append(pivot.loc[t, others].sum())

res_df['cross_asset_ofi'] = cross_vals

In [13]:
feature_df = (
    res_df
    .set_index(['symbol', 'timestamp'])
    .sort_index()
)

# Preview the first few rows
feature_df.head()

best_ofi  ofi_lvl_1  ofi_lvl_2  ofi_lvl_3  \
symbol timestamp                                                              
AAPL   2024-10-21 11:55:00+00:00    -870.0  -8.062329   3.104460   3.095193   
       2024-10-21 11:56:00+00:00   -1217.0 -11.204577  16.268273  16.765435   
       2024-10-21 11:57:00+00:00     334.0   3.047986  -5.229030  17.311465   
       2024-10-21 11:58:00+00:00     960.0   8.510998   5.789251   2.969984   
       2024-10-21 11:59:00+00:00     615.0   5.358388   0.156831   9.122328   

                                  ofi_lvl_4  ofi_lvl_5  ofi_lvl_6  ofi_lvl_7  \
symbol timestamp                                                               
AAPL   2024-10-21 11:55:00+00:00  -3.095193  -0.973040   4.123835  -1.538330   
       2024-10-21 11:56:00+00:00  -2.172786  -3.498553   6.794559  -7.264101   
       2024-10-21 11:57:00+00:00  -5.338538   5.949961  -2.482192  -0.584045   
       2024-10-21 11:58:00+00:00   1.817453 -15.647824   8.741504  -4.468274   
       2024-10-21 11:59:00+00:00   5.140567  -0.357226 -10.307272  10.376975   

                                  ofi_lvl_8  ofi_lvl_9  ofi_lvl_10  \
symbol timestamp                                                     
AAPL   2024-10-21 11:55:00+00:00  -3.716085   3.910693    1.251051   
       2024-10-21 11:56:00+00:00   1.482282  12.778926   -5.339897   
       2024-10-21 11:57:00+00:00   7.756851  -1.314102    2.308804   
       2024-10-21 11:58:00+00:00  -3.360071  14.778993   -2.083421   
       2024-10-21 11:59:00+00:00  -0.967124   4.940172   21.389986   

                                  integrated_ofi  cross_asset_ofi  
symbol timestamp                                                   
AAPL   2024-10-21 11:55:00+00:00       -6.601065              0.0  
       2024-10-21 11:56:00+00:00       -8.308474              0.0  
       2024-10-21 11:57:00+00:00      -19.617112              0.0  
       2024-10-21 11:58:00+00:00        6.902893              0.0  
       2024-10-21 11:59:00+00:00      -10.813791              0.0